In [13]:
FILE_PATH = "/Users/douglashindson/workspace/belief-state-machine/experiments/evals/finetuning_data/adam_and_eve/combined.txt"
BASE_MODEL = "gpt-4o-mini-2024-07-18" # "gpt-5-2024-08-06" # 
MODEL_RECORDS = "model_records_db.md"
SUMMARIZER_MODEL = "gpt-5-mini"


In [2]:
with open(FILE_PATH, 'r') as file:
    text = file.read()

text = text.split('\n\n')
print(f"loaded lines: {len(text)}")
print(text[0])
print(text[1])

loaded lines: 105
## The Stranger
The Home for Aged Persons is at Marengo, some fifty miles from Algiers. With the two o’clock bus I should get there well before nightfall. Then I can spend the night there, keeping the usual vigil beside the body, and be back here by tomorrow evening. I have fixed up with my employer for two days’ leave; obviously, under the circumstances, he couldn’t refuse. Still, I had an idea he looked annoyed, and I said, without thinking: “Sorry, sir, but it’s not my fault, you know.”


In [3]:
def extract_handmade_paragraph_pairs(text: list[str]) -> list[tuple[str, str]]:
    prev = None
    pairs = []
    for paragraph in text:
        if paragraph == '---':
            prev = None
            continue
        elif paragraph[:3] == "## ":
            prev = None
            continue
        elif prev == None:
            prev = paragraph
            continue
        elif paragraph == 'start_of_chapter':
            prev = "---"
            continue
        elif paragraph == 'start_of_piece':
            prev = ""
            continue
        elif paragraph[:2] == '\n':
            raise Exception(f"Bad formatting. Extra newline near {prev}")
        pairs.append((prev, paragraph))
        prev = paragraph
    return pairs

In [4]:
from pprint import pprint
raw_pairs = extract_handmade_paragraph_pairs(text)
print(f"loaded pairs: {len(raw_pairs)}")
pprint(raw_pairs[0])
pprint(raw_pairs[len(raw_pairs)//2])
pprint(raw_pairs[-1])

loaded pairs: 75
('The Home for Aged Persons is at Marengo, some fifty miles from Algiers. With '
 'the two o’clock bus I should get there well before nightfall. Then I can '
 'spend the night there, keeping the usual vigil beside the body, and be back '
 'here by tomorrow evening. I have fixed up with my employer for two days’ '
 'leave; obviously, under the circumstances, he couldn’t refuse. Still, I had '
 'an idea he looked annoyed, and I said, without thinking: “Sorry, sir, but '
 'it’s not my fault, you know.”',
 'Afterwards it struck me I needn’t have said that. I had no reason to excuse '
 'myself; it was up to him to express his sympathy and so forth. Probably he '
 'will do so the day after tomorrow, when he sees me in black. For the '
 'present, it’s almost as if Mother weren’t really dead. The funeral will '
 'bring it home to me, put an official seal on it, so to speak.')
('In a few days the Eldorado Expedition went into the patient wilderness, that '
 'closed upon it as t

In [5]:
import os
from openai import OpenAI

PROJECT_ID = "proj_hUizl3mrZGSfmp4C6DI60dJo"
client = OpenAI(
    # This is the default and can be omitted
    api_key=os.environ.get("OPENAI_API_KEY"),
)

In [6]:
def summarize_text(text: str):
    response = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": "Summarize this text extracted from a work of fiction. Each text should be summarized in 1-2 short sentences.",
        },
        {
            "role": "user",
            "content": text
        },
            
        ],
        model=SUMMARIZER_MODEL,
    )
    return response


In [7]:
from tqdm.notebook import tqdm
import pickle


USE_PICKLE = False

if USE_PICKLE:
    f = open('temp/finetune-checkpoint.pckl', 'rb')
    summaries = pickle.load(f)
    f.close()
else:
    summaries = []
    for pair in tqdm(raw_pairs, desc="Hitting chatgpt for summaries"):
        text_summary = summarize_text(pair[1])
        summaries.append(text_summary)
    f = open('temp/finetune-checkpoint.pckl', 'wb')
    pickle.dump(summaries, f)
    f.close()

Hitting chatgpt for summaries:   0%|          | 0/75 [00:00<?, ?it/s]

In [8]:
print(summaries[0].to_dict()['choices'][0]['message']['content'])

The narrator feels they didn't need to apologize and expects others to offer condolences later when they appear in mourning. For now the mother's death feels unreal, and the funeral will make it feel official.


In [9]:
def format_training_sample(prev_paragraph: str, summary: str, target_paragraph: str) -> dict:
    return {"messages": [
        {
            "role": "system",
            "content": "You are a science fiction writer. You will be provided a paragraph of a story and a summary of the paragraph that follows it. Use the preceding paragraph and the summary to write the paragraph that follows. The provided paragraph and summary are separated by \n---\n"
        },
        {
            "role": "user",
            "content": f"{prev_paragraph}\n---\n{summary}"
        },
        {
            "role": "assistant",
            "content": target_paragraph
        }
    ]}

In [10]:
import json

parsed_summaries = [summary.to_dict()['choices'][0]['message']['content'] for summary in summaries]
triples = [(pair[0], summary, pair[1]) 
           for pair, summary in zip(raw_pairs, parsed_summaries)]
training_filename = 'temp/summary-pair-tuning'
with open(training_filename, 'w') as f:
    for triple in triples:
        f.write(json.dumps(format_training_sample(triple[0], triple[1], triple[2])) + '\n')

In [11]:
file_response = client.files.create(
    file=open(training_filename, "rb"), purpose="fine-tune"
)
file_response

FileObject(id='file-Fp4ePNWtEMchtmTLTs6jxG', bytes=138114, created_at=1767284683, filename='summary-pair-tuning', object='file', purpose='fine-tune', status='processed', status_details=None, expires_at=None)

In [14]:
response = client.fine_tuning.jobs.create(model=BASE_MODEL, training_file=file_response.id)
print(response)

FineTuningJob(id='ftjob-jbQw0iQX0cGVgSDdPFTdwUTY', created_at=1767284978, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto'), model='gpt-4o-mini-2024-07-18', object='fine_tuning.job', organization_id='org-aKEzorvXA6tHQdC0x05ULMic', result_files=[], seed=1904073914, status='validating_files', trained_tokens=None, training_file='file-Fp4ePNWtEMchtmTLTs6jxG', validation_file=None, estimated_finish=None, integrations=[], method=Method(dpo=None, supervised=MethodSupervised(hyperparameters=MethodSupervisedHyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto')), type='supervised'), user_provided_suffix=None, metadata=None, usage_metrics=None, shared_with_openai=False, eval_id=None)


In [19]:
# Wait for this to finish

status_response = client.fine_tuning.jobs.retrieve(response.id)
print(f"STATUS: {status_response.status}")
print(f"MODEL ID: {status_response.fine_tuned_model}")
print(status_response)

STATUS: succeeded
MODEL ID: ft:gpt-4o-mini-2024-07-18:personal::CtFz8KYA
FineTuningJob(id='ftjob-jbQw0iQX0cGVgSDdPFTdwUTY', created_at=1767284978, error=Error(code=None, message=None, param=None), fine_tuned_model='ft:gpt-4o-mini-2024-07-18:personal::CtFz8KYA', finished_at=1767285520, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier=1.8, n_epochs=3), model='gpt-4o-mini-2024-07-18', object='fine_tuning.job', organization_id='org-aKEzorvXA6tHQdC0x05ULMic', result_files=['file-4KpdsF6GN5Je2Th2hKN6jh'], seed=1904073914, status='succeeded', trained_tokens=85794, training_file='file-Fp4ePNWtEMchtmTLTs6jxG', validation_file=None, estimated_finish=None, integrations=[], method=Method(dpo=None, supervised=MethodSupervised(hyperparameters=MethodSupervisedHyperparameters(batch_size=1, learning_rate_multiplier=1.8, n_epochs=3)), type='supervised'), user_provided_suffix=None, metadata=None, usage_metrics=None, shared_with_openai=False, eval_id=None)


In [20]:
# Test that the model does something

completion = client.chat.completions.create(
  model=status_response.fine_tuned_model,
  messages=[
    {"role": "system", "content": "You are a science fiction writer. Write the next paragraph."},
    {"role": "user", "content": "I was, am, will be... everything. Almost everynothing, life incarnate. Federations of federations of species, sprawling intra-dimensional compute-organisms evolved to higher and higher levels of consciousnesses. I am all of them, and I am searching. I am searching because I am always searching. I am not involved in the beginning or the end, but I am in every moment of time. I am simulating infinitely backwards and forwards, so I am in the moment and I am in the whole past and I am in the whole future, all at the same time. I am seeing through temporal boundaries, conquering new cardinalities of infinity, and existing across more planes of being than most beings can compute. I am practicing every religion, celebrating every culture, replaying every life I am able to live. I am finding..."}
  ]
)
pprint(completion.choices[0].message)

ChatCompletionMessage(content='... them, these signposts of the universe, milestones ringing in the mountain tops of improbable creation. I am finding that I adore them, whatever populations they occur to. At every undetermined node of decision, my self bifurcates anew and sees how this path mirrors that one, and what uncolonized planets they lead to. Omniscience, I am coming to learn, is fancies of a higher dimension. These pearls of soul, floating in the ever-madding sea of time and orbit and twist, dwell so close together that when a person opens her eyes upon one lifetime she can dimly sense the past lives the self has in other bodies, the future lives the self is moving into.', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None, annotations=[])


In [21]:
# Run this once the model has been fine-tuned to save it to the database

import datetime
with open(MODEL_RECORDS, 'a') as f:
    f.write(f"{str(datetime.date.today())}\n{FILE_PATH}\n{status_response.fine_tuned_model}\n")


In [22]:
print(status_response.fine_tuned_model)

ft:gpt-4o-mini-2024-07-18:personal::CtFz8KYA
